In [59]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from bertopic import BERTopic
from sklearn.linear_model import LinearRegression
import json

# Loading Embeddings

In [60]:
embeddings = np.load("embeddings.npy")

df = pd.read_parquet("documents.parquet")

# Model Decision

In [61]:
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

vectorizer = CountVectorizer(stop_words="english")
analyzer = vectorizer.build_analyzer()

In [62]:
tokenized_docs  = [analyzer(doc) for doc in df['text'].to_list()]

In [63]:
def compute_coherence(topic_model, docs, sample_size=None, coherence_type="c_v"):
    """
    Faster coherence computation for BERTopic
    """

    # 🔹 Sample kullan (çok önerilir)
    if sample_size is not None:
        docs = docs[:sample_size]

    # 🔹 Tokenize
    vectorizer = CountVectorizer(stop_words="english")
    analyzer = vectorizer.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in docs]

    # 🔹 Dictionary
    dictionary = Dictionary(tokenized_docs)

    # 🔹 Topic words al
    topics = topic_model.get_topics()
    topic_words = [
        [word for word, _ in topics[topic_id]]
        for topic_id in topics
        if topic_id != -1
    ]

    # 🔹 Coherence hesapla
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence=coherence_type
    )

    return coherence_model.get_coherence()

In [64]:
docs = df["text"].tolist()

results = []
models = {}
topics_dict = {} 
for min_size in [20, 50, 100]:

    model = BERTopic(
        min_topic_size=min_size,
        calculate_probabilities=False,
        verbose=True
    )

    topics, _ = model.fit_transform(docs, embeddings)

    score = compute_coherence(
        model,
        docs,
        sample_size=5000  # hız için sample
    )
    topic_count = len(model.get_topic_info()) - 1 
    results.append((min_size, score, topic_count))
    models[min_size] = model
    topics_dict[min_size] = topics

print(results)

2026-03-10 16:59:24,343 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-10 17:00:02,694 - BERTopic - Dimensionality - Completed ✓
2026-03-10 17:00:02,697 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-10 17:00:11,654 - BERTopic - Cluster - Completed ✓
2026-03-10 17:00:11,682 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-10 17:00:31,999 - BERTopic - Representation - Completed ✓
2026-03-10 17:01:38,057 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-10 17:02:16,305 - BERTopic - Dimensionality - Completed ✓
2026-03-10 17:02:16,307 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-10 17:02:25,059 - BERTopic - Cluster - Completed ✓
2026-03-10 17:02:25,080 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-10 17:02:45,797 - BERTopic - Representation - Completed ✓
2026-03-10 17:03:20,894 - BERTopic - D

[(20, np.float64(0.445794355781205), 728), (50, np.float64(0.4711016420230519), 350), (100, np.float64(0.5013437140325679), 211)]


In [65]:
df_results = pd.DataFrame(results, columns=["min_topic_size", "coherence", "topic_count"])
df_results

,min_topic_size,coherence,topic_count
0,20,0.445794,728
1,50,0.471102,350
2,100,0.501344,211


In [66]:
model = models[100]
topics = topics_dict[100]
reduced_model = model.reduce_topics(docs, nr_topics=100)
reduced_topics, _ = reduced_model.transform(docs, embeddings)

2026-03-10 17:05:03,422 - BERTopic - Topic reduction - Reducing number of topics
2026-03-10 17:05:03,516 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-10 17:05:25,837 - BERTopic - Representation - Completed ✓
2026-03-10 17:05:25,870 - BERTopic - Topic reduction - Reduced number of topics from 212 to 100
2026-03-10 17:05:29,832 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-10 17:05:30,229 - BERTopic - Dimensionality - Completed ✓
2026-03-10 17:05:30,230 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-10 17:05:51,712 - BERTopic - Cluster - Completed ✓


In [67]:
#model = BERTopic(
#    min_topic_size=100,
#    calculate_probabilities=False,
#    verbose=True,
#    nr_topics=100)

# Trend Analysis

In [68]:
df["topic"] = reduced_topics
df["month"] = df["published"].dt.to_period("M")

trend = (
    df[df.topic != -1] # -1 means outliers, we ignore them
    .groupby(["month", "topic"])
    .size()
    .unstack(fill_value=0)
)

trend_share = trend.div(trend.sum(axis=1), axis=0)

In [69]:
def compute_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values

        if y.sum() == 0:
            continue

        model = LinearRegression()
        model.fit(x, y)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)

def compute_log_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values



        y_log = np.log(y + 1e-6)

        model = LinearRegression()
        model.fit(x, y_log)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)



In [70]:
log_trend_scores_12 = compute_log_trend_scores(trend_share, window=12)

print("🔥 Emerging topics:")
print(log_trend_scores_12.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_12.tail(10))

🔥 Emerging topics:
85    0.271307
18    0.073102
44    0.065363
60    0.063242
42    0.047986
87    0.047861
98    0.044810
48    0.037099
79    0.036507
2     0.034787
dtype: float64

📉 Declining topics:
82   -0.122420
91   -0.142607
80   -0.279942
90   -0.280391
77   -0.292046
52   -0.302698
78   -0.307222
34   -0.330372
86   -0.334358
76   -0.419938
dtype: float64


In [71]:
log_trend_scores_24 = compute_log_trend_scores(trend_share, window=24)

print("🔥 Emerging topics:")
print(log_trend_scores_24.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_24.tail(10))

🔥 Emerging topics:
81    0.085359
18    0.068070
2     0.064675
44    0.037164
79    0.025946
37    0.025371
29    0.025162
73    0.024134
62    0.022138
48    0.022088
dtype: float64

📉 Declining topics:
31   -0.063961
89   -0.066679
78   -0.081057
94   -0.086701
77   -0.090827
56   -0.093943
72   -0.102116
76   -0.108285
52   -0.121183
34   -0.138697
dtype: float64


In [72]:
len(log_trend_scores_12), len(log_trend_scores_24)

(90, 94)

#  Visualization

In [73]:
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler


In [74]:
def clean_label(name):
    parts = name.split("_")[2:]  # topic_x_ kısmını at
    return " ".join(parts[:3])   # ilk 3 keyword yeterli

def generate_label(topic_id, top_n=3):
    words = [word for word, _ in reduced_model.get_topic(topic_id)]
    return " / ".join(words[:top_n])

def assign_quadrant(row):
    if row["slope_24m"] > 0 and row["slope_12m"] > 0:
        return "🚀 Strong & Accelerating"
    
    elif row["slope_24m"] > 0 and row["slope_12m"] < 0:
        return "📈 Strong but Slowing"
    
    elif row["slope_24m"] < 0 and row["slope_12m"] > 0:
        return "🌱 Emerging"
    
    else:
        return "📉 Declining"

In [75]:
topics_df  = reduced_model.get_topic_info()
topics_df = topics_df[topics_df.Topic != -1]

In [76]:
topics_df["label"] = topics_df["Name"].apply(clean_label)
topics_df["label"] = topics_df["label"].str.capitalize()

In [77]:
slope_df =pd.DataFrame({
    "slope_12m": log_trend_scores_12,
    "slope_24m": log_trend_scores_24
})
slope_df["Topic"] = slope_df.index

topics_df = topics_df.merge(slope_df, on="Topic")
topics_df.set_index("Topic", inplace=True)


In [78]:
topic_map = topics_df[["label"]].to_dict()["label"]

In [79]:
topics_df["category"] = topics_df.apply(assign_quadrant, axis=1)
topics_df["acceleration"] = topics_df["slope_12m"] - topics_df["slope_24m"]

In [80]:
topics_df["growth_score"] = (topics_df["slope_12m"] * np.log1p(topics_df["Count"]))
scaler = MinMaxScaler()
topics_df["growth_size"] = scaler.fit_transform(topics_df[["growth_score"]])
topics_df["growth_size"] = topics_df["growth_size"] * 40 + 10

In [81]:
topics_df.head()

,Count,Name,Representation,Representative_Docs,label,slope_12m,slope_24m,category,acceleration,growth_score,growth_size
Topic,,,,,,,,,,,
0,7435,0_speech_audio_asr_recognition,"[speech, audio, asr, recognition, music, speak...",[EMOVIE: A Mandarin Emotion Speech Dataset wit...,Audio asr recognition,-0.005205,-0.001793,📉 Declining,-0.003412,-0.046396,34.843398
1,5362,1_explanations_preference_models_to,"[explanations, preference, models, to, of, the...",[Multi-Domain Explainability of Preferences Pr...,Preference models to,-0.017420,-0.018682,📉 Declining,0.001263,-0.149589,33.727941
2,4132,2_reasoning_cot_mathematical_llms,"[reasoning, cot, mathematical, llms, models, t...",[Lost at the Beginning of Reasoning Recent adv...,Cot mathematical llms,0.034787,0.064675,🚀 Strong & Accelerating,-0.029889,0.289660,38.475939
3,3715,3_code_llms_software_safety,"[code, llms, software, safety, jailbreak, lang...",[Re-Evaluating Code LLM Benchmarks Under Seman...,Llms software safety,0.017969,0.016862,🚀 Strong & Accelerating,0.001108,0.147716,36.941618
4,3679,4_retrieval_rag_question_knowledge,"[retrieval, rag, question, knowledge, answerin...",[Parametric Retrieval Augmented Generation Ret...,Rag question knowledge,-0.028608,-0.014650,📉 Declining,-0.013958,-0.234888,32.805916


In [82]:
topics_df["label"].nunique()

94

In [83]:
def get_topic_words(topic_id, top_n=10, model=reduced_model):
    
    words = model.get_topic(topic_id)
    
    words = words[:top_n]
    
    terms = [w[0] for w in words]
    scores = [w[1] for w in words]
    
    return terms, scores

In [84]:
topic_words = {}
for topic_id in topics_df.index:
    terms, scores = get_topic_words(topic_id)
    topic_words[topic_id] = (terms, scores)  

In [85]:
with open('topic_words.json', 'w') as f:
    json.dump(topic_words, f)

In [86]:
topics_df.to_csv("topics_trend_analysis.csv")
trend_share.to_csv("trend_share.csv")


In [87]:
top_topics = topics_df["slope_12m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 12-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [88]:
top_topics = topics_df["slope_24m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 24-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [89]:
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",

    title="AI-Powered Research Trend Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [90]:
top_topics = topics_df[(topics_df["slope_12m"] > 0)].sort_values(by="acceleration", ascending=False).head(10).index
topics_df = topics_df.loc[top_topics]
    
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",
    title="AI-Powered Research Trend Quadrant",
)


fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [91]:
df_plot = topics_df.dropna()
df_plot["category"] = df_plot.apply(assign_quadrant, axis=1)
fig = px.scatter(
    df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [92]:
top10_growth = (topics_df.sort_values("growth_score", ascending=False).head(10).index)

topics_df_plot = topics_df.loc[top10_growth]
fig = px.scatter(
    topics_df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [93]:
impact_df = topics_df[(topics_df['slope_12m'] > 0)&
                    (topics_df["acceleration"] > 0)].copy()

impact_df = impact_df.sort_values('growth_score', ascending=False)

top10_impact = impact_df.head(10)

In [94]:
top10_impact.to_csv("top10_impact_topics.csv")

In [95]:
import pandas as pd
pd.read_csv("top10_impact_topics.csv", index_col=0)

,Count,Name,Representation,Representative_Docs,label,slope_12m,slope_24m,category,acceleration,growth_score,growth_size
Topic,,,,,,,,,,,
85,147,85_food_recipes_recipe_dietary,"['food', 'recipes', 'recipe', 'dietary', 'ingr...",['Diffusion Model with Clustering-based Condit...,Recipes recipe dietary,0.271307,-0.013493,🌱 Emerging,0.284800,1.355779,50.000000
44,756,44_diffusion_generation_sampling_models,"['diffusion', 'generation', 'sampling', 'model...","[""Energy-Based Diffusion Language Models for T...",Generation sampling models,0.065363,0.037164,🚀 Strong & Accelerating,0.028199,0.433315,40.028762
60,434,60_bandit_regret_bandits_algorithm,"['bandit', 'regret', 'bandits', 'algorithm', '...","[""Contextual Bandits and Imitation Learning vi...",Regret bandits algorithm,0.063242,-0.005001,🌱 Emerging,0.068243,0.384220,39.498071
42,889,42_eeg_brain_signals_decoding,"['eeg', 'brain', 'signals', 'decoding', 'bci',...",['A Simple Review of EEG Foundation Models: Da...,Brain signals decoding,0.047986,0.001750,🚀 Strong & Accelerating,0.046235,0.325882,38.867473
87,131,87_patent_patents_technology_intellectual,"['patent', 'patents', 'technology', 'intellect...",['Natural Language Processing in the Patent Do...,Patents technology intellectual,0.047861,-0.010028,🌱 Emerging,0.057889,0.233697,37.871017
98,103,98_vr_reality_ar_virtual,"['vr', 'reality', 'ar', 'virtual', 'xr', 'user...",['Transcending Dimensions using Generative AI:...,Reality ar virtual,0.044810,0.003546,🚀 Strong & Accelerating,0.041264,0.208116,37.594509
70,300,70_voting_auction_bidding_auctions,"['voting', 'auction', 'bidding', 'auctions', '...",['Online Fair Division for Personalized $2$-Va...,Auction bidding auctions,0.032298,-0.002365,🌱 Emerging,0.034664,0.184331,37.337404
46,718,46_privacy_private_dp_differential,"['privacy', 'private', 'dp', 'differential', '...",['Synthetic Text Generation with Differential ...,Private dp differential,0.027315,-0.002492,🌱 Emerging,0.029807,0.179675,37.287070
38,943,38_causal_variables_discovery_causality,"['causal', 'variables', 'discovery', 'causalit...",['A Survey on Causal Discovery Methods for I.I...,Variables discovery causality,0.021788,-0.005863,🌱 Emerging,0.027651,0.149249,36.958186
